# Results Presentation HTML

Build a portable static HTML report in `/host/d/projects/Habitats/results`. Re-run this notebook after updating result figures or final selections.

In [1]:

# ============================================================
# Build portable HTML results presentation
# ============================================================

import os
import json
import subprocess
from pathlib import Path
from html import escape

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/host/d/Github/Osteosarcoma')
RESULTS_DIR = Path('/host/d/projects/Habitats/results')
ASSET_DIR = RESULTS_DIR / 'html_assets'
MODEL_ROOT = Path('/host/d/projects/Habitats/models/Prognosis')
RADIOMICS_ROOT = Path('/host/d/projects/Habitats/radiomics')
PATIENT_LIST_FILE = Path('/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random0.xlsx')
HTML_OUT = RESULTS_DIR / 'results_presentation.html'

ASSET_DIR.mkdir(parents=True, exist_ok=True)

TITLE = 'Integration of multi-scale radiomics and deep learning for three-year prognosis of osteosarcoma'
ML_METHODS = 'SVM, LR, RF, and XGBoost'


def read_excel_if_exists(path, **kwargs):
    path = Path(path)
    if not path.exists():
        return None
    return pd.read_excel(path, **kwargs)


def read_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def rel_to_results(path):
    path = Path(path)
    return os.path.relpath(path, RESULTS_DIR).replace(os.sep, '/')


def pdf_to_png(pdf_path, asset_name=None, dpi=180):
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        return None
    if asset_name is None:
        asset_name = pdf_path.stem
    png_path = ASSET_DIR / f'{asset_name}.png'
    try:
        subprocess.run(
            ['pdftoppm', '-png', '-singlefile', '-r', str(dpi), str(pdf_path), str(png_path.with_suffix(''))],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )
        return png_path
    except Exception as exc:
        print('PDF preview conversion failed:', pdf_path, repr(exc))
        return None


def image_card(pdf_filename, caption='', asset_name=None, css_class='figure-card'):
    pdf_path = RESULTS_DIR / pdf_filename
    if not pdf_path.exists():
        return f'<div class="missing">Missing figure: {escape(pdf_filename)}</div>'
    png_path = pdf_to_png(pdf_path, asset_name=asset_name or Path(pdf_filename).stem)
    href = rel_to_results(pdf_path)
    if png_path is not None:
        src = rel_to_results(png_path)
        img = f'<a href="{href}" target="_blank"><img src="{src}" loading="lazy" /></a>'
    else:
        img = f'<a class="pdf-link" href="{href}" target="_blank">Open PDF: {escape(pdf_filename)}</a>'
    cap = f'<figcaption>{caption}</figcaption>' if caption else ''
    return f'<figure class="{css_class}">{img}{cap}</figure>'


def fmt_num(x, digits=3):
    if pd.isna(x):
        return ''
    try:
        return f'{float(x):.{digits}f}'
    except Exception:
        return str(x)


def is_blank_value(value):
    if value is None:
        return True
    try:
        if pd.isna(value):
            return True
    except Exception:
        pass
    if isinstance(value, str) and value.strip().lower() in {'nan', 'none', 'nat'}:
        return True
    return False


def is_p_value_column(col):
    name = str(col).lower().replace('-', '_').replace(' ', '_')
    return (
        'p_value' in name
        or name in {'p', 'pvalue', 'p_val', 'pvalue_formatted'}
        or name.endswith('_p')
    )


def is_significant_p(value, threshold=0.05):
    if is_blank_value(value):
        return False
    text = str(value).strip().lower()
    text = text.replace('p', '').replace('=', '').replace(' ', '')
    try:
        if text.startswith('<'):
            return float(text[1:]) < threshold
        if text.startswith('≤') or text.startswith('<='):
            text = text.replace('≤', '').replace('<=', '')
            return float(text) <= threshold
        return float(text) < threshold
    except Exception:
        return False


def format_table_value(value):
    if is_blank_value(value):
        return ''
    if isinstance(value, (float, np.floating)):
        return fmt_num(value, 3)
    return str(value)


def df_to_html_table(df, max_rows=None, classes='data-table compact'):
    if df is None:
        return '<div class="missing">Table not found.</div>'
    out = df.copy()
    if max_rows is not None:
        out = out.head(max_rows)

    html = [f'<table class="{classes}">', '<thead><tr>']
    for col in out.columns:
        html.append(f'<th>{escape(str(col))}</th>')
    html.append('</tr></thead><tbody>')

    p_cols = {col for col in out.columns if is_p_value_column(col)}
    for _, row in out.iterrows():
        html.append('<tr>')
        for col in out.columns:
            value = row[col]
            display = format_table_value(value)
            cls = ' class="p-sig"' if col in p_cols and is_significant_p(value) else ''
            html.append(f'<td{cls}>{escape(display)}</td>')
        html.append('</tr>')
    html.append('</tbody></table>')
    return ''.join(html)


def nonfeature_cols():
    return {
        'Patient_set', 'Patient_index', 'Image_filepath', 'Mask_filepath',
        'Prognosis_label', 'Pathologic_label', 'split', 'fold', 'Label', 'label',
        'k', 'Habitat_pixel_num', 'Total_habitat_pixel_num', 'Habitat_pixel_fraction',
    }


def count_feature_columns(path):
    df = read_excel_if_exists(path, nrows=2)
    if df is None:
        return None
    return len([c for c in df.columns if c not in nonfeature_cols()])


def get_final_dir(model_key):
    mapping = {
        'clinical': MODEL_ROOT / 'clinical' / 'final_selections',
        'whole_image': MODEL_ROOT / 'whole_image' / 'final_selections',
        'habitats_sum': MODEL_ROOT / 'habitats_sum' / 'final_selections',
        'dl_3d_ml_all': MODEL_ROOT / 'dl_3d_ml_all' / 'final_selections',
        'fusion_stacking': MODEL_ROOT / 'fusion' / 'stacking' / 'final_selections' / 'RF',
    }
    return mapping[model_key]


def manifest_summary(model_key):
    root = get_final_dir(model_key)
    data = {}
    for cohort, fname in [
        ('cv', 'cv_final_selection_manifest.json'),
        ('internal_test', 'internal_test_final_selection_manifest.json'),
        ('external_test', 'external_test_final_selection_manifest.json'),
        ('train', 'train_selected_settings_manifest.json'),
    ]:
        data[cohort] = read_json_if_exists(root / fname)
    return data


def metric_row(root, fname):
    df = read_excel_if_exists(root / fname)
    if df is None or df.empty:
        return {}
    if 'is_final_selection' in df.columns and df['is_final_selection'].astype(bool).any():
        row = df[df['is_final_selection'].astype(bool)].iloc[-1]
    else:
        row = df.iloc[-1]
    return row.to_dict()


def selected_experiments_text(manifest, key_candidates):
    if not manifest:
        return 'N/A'
    items = []
    for key in key_candidates:
        if key in manifest:
            items = manifest[key]
            break
    if not items:
        return 'N/A'
    parts = []
    for item in items:
        clf = item.get('classifier', '')
        exp = item.get('experiment', '')
        parts.append(f'{clf} ({exp})'.strip())
    return '; '.join(parts)


def selected_feature_file(classifier, experiment, select_dir):
    select_dir = Path(select_dir)
    tokens = str(experiment).split('_')
    pattern = f'*{classifier}*Prognosis*' + '*'.join(tokens) + '*_selected.xlsx'
    matches = sorted(select_dir.glob(pattern))
    if matches:
        return matches[0]
    pattern = f'*{classifier}*{experiment}*_selected.xlsx'
    matches = sorted(select_dir.glob(pattern))
    if matches:
        return matches[0]
    return None


def selected_counts_for_manifest(manifest, select_dir):
    if not manifest or 'selected_cv_experiments' not in manifest:
        return []
    rows = []
    for item in manifest['selected_cv_experiments']:
        clf = item.get('classifier')
        exp = item.get('experiment')
        path = selected_feature_file(clf, exp, select_dir)
        rows.append({
            'Classifier': clf,
            'Experiment': exp,
            'Selected features': count_feature_columns(path) if path else None,
            'Selected feature file': path.name if path else 'Not found',
        })
    return rows


def feature_rows_html(rows):
    if not rows:
        return '<p class="muted">No selected-feature file found.</p>'
    # Method sections only show the first CV-selected setting, even when final selection used several settings.
    return df_to_html_table(pd.DataFrame(rows[:1]), classes='data-table mini')


def fig_grid(items, cls='grid two'):
    return f'<div class="{cls}">' + '\n'.join(items) + '</div>'

# Cohort table
split_df = pd.read_excel(PATIENT_LIST_FILE)
label_col = 'Prognosis_label'
cohort_rows = []
for cohort_name, folds in [('Training', [0,1,2,3,4]), ('Validation', [5]), ('Testing', [6])]:
    sub = split_df[split_df['fold'].isin(folds)].copy()
    cohort_rows.append({
        'Cohort': cohort_name,
        'Fold(s)': ','.join(str(f) for f in folds),
        'Cases': int(sub.shape[0]),
        'Label=1, n': int(sub[label_col].sum()),
        'Label=1, %': f'{100 * sub[label_col].mean():.1f}%',
    })
cohort_df = pd.DataFrame(cohort_rows)

clinical_vars = [
    'Age', 'Sex', 'Lesion_site', 'Pathologic_fracture', 'Height_at_visit', 'Weight_at_visit', 'BMI',
    'WBC', 'HGB', 'PLT', 'CRP', 'ALP', 'Total_cholesterol', 'Triglycerides', 'LDL', 'LDH',
    'PT', 'APTT', 'Fibrinogen', 'D-dimer',
    'Tumor_AP_diameter_mm', 'Tumor_longitudinal_diameter_mm', 'Tumor_transverse_diameter_mm', 'Tumor_volume_mm3',
]

final_manifests = {key: manifest_summary(key) for key in ['clinical', 'whole_image', 'habitats_sum', 'dl_3d_ml_all', 'fusion_stacking']}
feature_info = {
    'Clinical': selected_counts_for_manifest(final_manifests['clinical']['cv'], RADIOMICS_ROOT / 'clinical_variables' / 'select'),
    'C-radiomics': selected_counts_for_manifest(final_manifests['whole_image']['cv'], RADIOMICS_ROOT / 'whole_image' / 'select'),
    'H-radiomics': selected_counts_for_manifest(final_manifests['habitats_sum']['cv'], RADIOMICS_ROOT / 'habitats' / 'select_sum'),
    'DL_3D-ML': selected_counts_for_manifest(final_manifests['dl_3d_ml_all']['cv'], RADIOMICS_ROOT / 'dl_3d_ml_all' / 'select'),
}

method_summary_rows = []
for model_label, key in [('Clinical', 'clinical'), ('C-radiomics', 'whole_image'), ('H-radiomics', 'habitats_sum'), ('DL_3D-ML', 'dl_3d_ml_all'), ('Fusion stacking', 'fusion_stacking')]:
    cv_manifest = final_manifests[key]['cv']
    int_manifest = final_manifests[key]['internal_test']
    ext_manifest = final_manifests[key]['external_test']
    method_summary_rows.append({
        'Model': model_label,
        'CV selected setting(s)': selected_experiments_text(cv_manifest, ['selected_cv_experiments']),
        'CV probability': cv_manifest.get('final_selection_source_probability_column', '') if cv_manifest else '',
        'Validation selected setting(s)': selected_experiments_text(int_manifest, ['selected_internal_test_experiments']),
        'Validation probability': int_manifest.get('final_selection_source_probability_column', '') if int_manifest else '',
        'Testing selected setting(s)': selected_experiments_text(ext_manifest, ['selected_external_test_experiments']),
        'Testing probability': ext_manifest.get('final_selection_source_probability_column', '') if ext_manifest else '',
    })
method_summary_df = pd.DataFrame(method_summary_rows)

whole_initial = count_feature_columns(RADIOMICS_ROOT / 'whole_image' / 'radiomics_measurements.xlsx')
whole_pcc = count_feature_columns(RADIOMICS_ROOT / 'whole_image' / 'radiomics_measurements_PCC.xlsx')
hab_initial = count_feature_columns(RADIOMICS_ROOT / 'habitats' / 'habitat_radiomics_measurements_sum_normalized.xlsx')
hab_pcc = count_feature_columns(RADIOMICS_ROOT / 'habitats' / 'habitat_radiomics_measurements_sum_PCC.xlsx')

raw_dl_feature = RADIOMICS_ROOT / 'dl_3d_ml_all' / 'features_numpy' / 'all_DLfeature_raw_selected.npy'
pca_info_path = RADIOMICS_ROOT / 'dl_3d_ml_all' / 'features_numpy' / 'pca_info.json'
dl_raw_n = None
if raw_dl_feature.exists():
    try:
        dl_raw_n = int(np.load(raw_dl_feature, mmap_mode='r').shape[1])
    except Exception:
        dl_raw_n = None
pca_info = read_json_if_exists(pca_info_path) or {}
dl_pca_n = count_feature_columns(RADIOMICS_ROOT / 'dl_3d_ml_all' / 'dl_3d_features_PCA.xlsx')
explained = pca_info.get('explained_variance_ratio_sum', pca_info.get('explained_variance_sum', 'N/A'))
try:
    explained_txt = f'{float(explained):.3f}'
except Exception:
    explained_txt = str(explained)

perf_rows = []
metric_files = {
    'Train': 'train_selected_settings_metrics.xlsx',
    'Validation': 'internal_test_final_selection_metrics.xlsx',
    'Testing': 'external_test_final_selection_metrics.xlsx',
}
for display_model, key in [('Clinical', 'clinical'), ('C-radiomics', 'whole_image'), ('H-radiomics', 'habitats_sum'), ('DL_3D', 'dl_3d_ml_all'), ('Fusion stacking', 'fusion_stacking')]:
    root = get_final_dir(key)
    for cohort, fname in metric_files.items():
        row = metric_row(root, fname)
        if row:
            perf_rows.append({
                'Cohort': cohort,
                'Model': display_model,
                'AUC': row.get('auc', np.nan),
                '95% CI': f"{fmt_num(row.get('auc_ci_low'))}-{fmt_num(row.get('auc_ci_high'))}",
                'Accuracy': row.get('accuracy', np.nan),
                'Sensitivity': row.get('sensitivity', np.nan),
                'Specificity': row.get('specificity', np.nan),
            })
perf_df = pd.DataFrame(perf_rows)

demo_df = read_excel_if_exists(RESULTS_DIR / 'demographics_table_prognosis.xlsx')
logistic_df = read_excel_if_exists(RESULTS_DIR / 'clinical_univariate_multivariate_logistic_table_prognosis.xlsx')
if demo_df is not None:
    demo_display = demo_df.copy()
    demo_display = demo_display.drop(columns=[c for c in ['Variable_type'] if c in demo_display.columns])
else:
    demo_display = None
logistic_display = logistic_df.copy() if logistic_df is not None else None
if logistic_display is not None:
    logistic_display = logistic_display.drop(columns=[c for c in ['Comparison_or_unit'] if c in logistic_display.columns])

css = '''
:root { --ink:#1b1b1d; --muted:#64666b; --line:#d9dde5; --soft:#f5f7fa; --accent:#2f6f9f; --accent-soft:#eaf3f8; }
* { box-sizing: border-box; }
body { margin:0; background:#eef1f5; color:var(--ink); font-family:"Times New Roman", Times, "Noto Serif CJK SC", SimSun, serif; font-size:15px; line-height:1.45; }
.page { max-width:1260px; margin:0 auto; background:#fff; min-height:100vh; padding:34px 42px 58px; }
h1 { margin:0 0 10px; font-size:31px; line-height:1.12; font-weight:700; text-transform:capitalize; }
.subtitle { color:var(--muted); margin:0 0 24px; font-size:15px; }
h2 { margin:28px 0 12px; padding-top:12px; border-top:2px solid #222; font-size:22px; line-height:1.2; }
h3 { margin:20px 0 8px; font-size:18px; }
p { margin:7px 0 10px; }
.muted { color:var(--muted); }
.data-table { width:100%; border-collapse:collapse; margin:8px 0 14px; font-size:13px; }
.data-table th { background:#25313f; color:white; font-weight:700; padding:6px 8px; border:1px solid #25313f; text-align:left; white-space:nowrap; }
.data-table td { border:1px solid var(--line); padding:5px 8px; vertical-align:top; }
.data-table tr:nth-child(even) td { background:#fafbfc; }
.p-sig { color:#c00000; font-weight:700; }
.data-table.compact td, .data-table.compact th { padding:4px 6px; }
.data-table.mini { font-size:12.5px; }
.table-scroll { overflow-x:auto; border:1px solid var(--line); padding:0 0 2px; margin:8px 0 14px; }
.grid { display:grid; gap:14px; align-items:start; margin:10px 0 16px; }
.grid.two { grid-template-columns:repeat(2, minmax(0, 1fr)); }
.figure-card { margin:0; border:1px solid var(--line); background:#fff; padding:8px; }
.figure-card img { display:block; width:100%; max-height:720px; object-fit:contain; }
.figure-card figcaption { margin-top:6px; font-size:12.5px; color:var(--muted); }
.figure-card.wide img { max-height:900px; }
.pdf-link { display:block; padding:16px; border:1px dashed var(--line); }
.method-box { border:1px solid var(--line); padding:12px 14px; background:#fff; margin:10px 0 14px; }
.missing { color:#a33; border:1px dashed #e3b4b4; background:#fff6f6; padding:10px; }
@media print { body { background:#fff; } .page { padding:18px; max-width:none; } .figure-card { break-inside:avoid; } }
@media (max-width:900px) { .page { padding:22px 16px; } .grid.two { grid-template-columns:1fr; } }
'''

html_parts = []
html_parts.append(f'''<!doctype html>
<html lang="zh-CN">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>{escape(TITLE)}</title>
<style>{css}</style>
</head>
<body>
<main class="page">
<h1>{escape(TITLE)}</h1>
<p class="subtitle">Static results summary. Figures are shown as portable previews; click each figure to open the original PDF.</p>
''')

html_parts.append('''
<h2>1. Cohort</h2>
<p>本研究按三年预后标签（<em>Prognosis_label</em>）建模。当前数据划分为 training、validation 和 testing 三个 cohort，其中 validation 对应 internal test，testing 对应 external test。</p>
''')
html_parts.append(df_to_html_table(cohort_df))

html_parts.append(f'''
<h2>2. Methods</h2>
<p>所有机器学习分支均评估四类分类器：<strong>{ML_METHODS}</strong>。每个单模态模型先完成各自的 feature selection 和模型选择，再进入 final selection；融合模型使用各单模态模型的阳性概率作为输入。</p>
''')

clinical_var_list = ', '.join(clinical_vars)
html_parts.append(f'''
<div class="method-box">
<h3>2.2 Clinical model</h3>
<p>Clinical model 使用 {len(clinical_vars)} 个候选 clinical variables，包括：{escape(clinical_var_list)}。缺失值和分类变量在 clinical feature workflow 中完成处理，随后进入 ML pipeline。最终 selected features 根据 CV final selection 中的第一个 setting 读取。</p>
{feature_rows_html(feature_info['Clinical'])}
</div>
''')

html_parts.append(f'''
<div class="method-box">
<h3>2.3 C-radiomics</h3>
<p>C-radiomics 指 whole-tumor conventional radiomics。所有病例使用原始 tumor mask 提取 whole-tumor radiomics features；初始特征数为 <strong>{whole_initial if whole_initial is not None else 'N/A'}</strong>，经过 ICC/PCC 相关性筛选后剩余 <strong>{whole_pcc if whole_pcc is not None else 'N/A'}</strong> 个特征。随后使用 ML selector 在 final selected setting 中进一步筛选。</p>
{feature_rows_html(feature_info['C-radiomics'])}
</div>
''')

html_parts.append(f'''
<div class="method-box">
<h3>2.4 H-radiomics</h3>
<p>H-radiomics 基于 voxel-level radiomics feature maps 构建。先对预设 voxel radiomics features 进行 per-case normalization，再用 K-means 生成 tumor habitats；cohort-level silhouette coefficient (SC) 和 Calinski-Harabasz (CH) index 支持选择固定 <strong>K=4</strong>。每个 habitat 单独提取 radiomics features，并以 sum 方式汇总为 H-radiomics feature table。初始 sum features 为 <strong>{hab_initial if hab_initial is not None else 'N/A'}</strong>，PCC 后为 <strong>{hab_pcc if hab_pcc is not None else 'N/A'}</strong>。</p>
{feature_rows_html(feature_info['H-radiomics'])}
</div>
''')
html_parts.append(fig_grid([
    image_card('habitat_K_selection_SC_CH.pdf', 'Cohort-level SC and CH curves used for fixed K selection.'),
    image_card('habitat_representative_set_1_1_slice13.pdf', 'Representative tumor mask and K=4 habitat partition.'),
], cls='grid two'))

html_parts.append(f'''
<div class="method-box">
<h3>2.5 DL_3D-ML</h3>
<p>DL_3D branch 使用 3D ResNet18 作为 feature extractor。输入为 <strong>96 x 96 x 64</strong> ROI volume，并组织成 3 个语义通道：full context、bbox-only 和 tumor-only。模型的 penultimate-layer features 作为 DL features，随后使用 PCA 降维；当前 raw DL feature 数为 <strong>{dl_raw_n if dl_raw_n is not None else 'N/A'}</strong>，PCA 后特征数为 <strong>{dl_pca_n if dl_pca_n is not None else 'N/A'}</strong>，解释方差约为 <strong>{explained_txt}</strong>。PCA features 再进入和 radiomics 相同的 ML pipeline。</p>
{feature_rows_html(feature_info['DL_3D-ML'])}
</div>
''')
html_parts.append(fig_grid([
    image_card('gradcam_3d_positive_set_2_58.pdf', 'Positive representative Grad-CAM. Left: MRI with tumor mask; middle: Grad-CAM; right: overlay.'),
    image_card('gradcam_3d_negative_set_1_145.pdf', 'Negative representative Grad-CAM. Grad-CAM target class is the negative class.'),
], cls='grid two'))

html_parts.append('''
<div class="method-box">
<h3>2.6 Fusion</h3>
<p>Fusion 使用四个输入概率：clinical、C-radiomics、H-radiomics 和 DL_3D。<strong>Soft_vote</strong> 直接计算四个概率的算术平均。<strong>Stacking</strong> 将四个概率作为 meta-features，并使用 ML meta-learner 进行二级建模；当前 final stacking model 为 RF。</p>
</div>
''')

html_parts.append('''
<h2>3. Results</h2>
<h3>3.1 Demographics</h3>
<p>连续变量以 mean ± SD 汇报，分类变量以 n (%) 汇报。每个 cohort 内分别比较 label=0 与 label=1。</p>
<div class="table-scroll">
''')
html_parts.append(df_to_html_table(demo_display, classes='data-table compact'))
html_parts.append('</div>')

html_parts.append('''
<h3>3.2 Clinical univariate and multivariate logistic regression</h3>
<p>Univariate logistic regression 对每个 clinical variable 单独建模；multivariate logistic regression 将候选变量放入同一个模型中。OR 表示 label=1 odds 的相对变化，括号内为 95% CI。</p>
<div class="table-scroll">
''')
html_parts.append(df_to_html_table(logistic_display, classes='data-table compact'))
html_parts.append('</div>')

html_parts.append('''
<h3>3.3 Model performance</h3>
<p>Performance heatmap 汇总 training、validation 和 testing cohorts 中各模型的 AUC、accuracy、sensitivity 和 specificity。</p>
''')
html_parts.append(image_card('performance_heatmap.pdf', 'Final selected model performance across cohorts.', css_class='figure-card wide'))
html_parts.append('''
<h3>3.4 ROC and DCA</h3>
<p>ROC curves compare discrimination; DCA curves compare threshold-dependent clinical net benefit.</p>
''')
html_parts.append(image_card('ROC_DCA_combined.pdf', 'ROC and decision curve analysis across the three cohorts.', css_class='figure-card wide'))
html_parts.append('''
<h3>3.5 DeLong test</h3>
<p>Pairwise DeLong tests compare AUC differences between final selected models.</p>
''')
html_parts.append(image_card('DeLong_combined.pdf', 'Pairwise DeLong p-value heatmaps.', css_class='figure-card wide'))
html_parts.append('''
<h3>3.6 Survival analysis</h3>
<p>Kaplan-Meier curves stratify patients into high- and low-risk groups using model-derived probability thresholds.</p>
''')
html_parts.append(image_card('KM_combined.pdf', 'Kaplan-Meier curves for final selected models.', css_class='figure-card wide'))

html_parts.append('''
<h3>3.7 Fusion model interpretability</h3>
<p>SHAP analysis explains the RF stacking model at the probability-fusion level. The four meta-features are clinical, C-radiomics, H-radiomics and DL_3D probabilities.</p>
''')
html_parts.append(image_card('fusion_stacking_SHAP_summary.pdf', 'Global SHAP summary for RF stacking fusion.', css_class='figure-card wide'))
html_parts.append(fig_grid([
    image_card('fusion_stacking_MRI_overlay_positive_set_2_92.pdf', 'Positive case MRI with tumor overlay.'),
    image_card('fusion_stacking_SHAP_waterfall_positive.pdf', 'Positive case SHAP waterfall.'),
    image_card('fusion_stacking_MRI_overlay_negative_set_2_107.pdf', 'Negative case MRI with tumor overlay.'),
    image_card('fusion_stacking_SHAP_waterfall_negative.pdf', 'Negative case SHAP waterfall.'),
], cls='grid two'))

html_parts.append('''
</main>
</body>
</html>
''')

HTML_OUT.write_text('\n'.join(html_parts), encoding='utf-8')

print('Wrote HTML:', HTML_OUT)
print('Assets:', ASSET_DIR)
print('Number of assets:', len(list(ASSET_DIR.glob('*'))))
print('\nCohort table:')
print(cohort_df.to_string(index=False))
print('\nMethod summary:')
print(method_summary_df.to_string(index=False))


Wrote HTML: /host/d/projects/Habitats/results/results_presentation.html
Assets: /host/d/projects/Habitats/results/html_assets
Number of assets: 13

Cohort table:
    Cohort   Fold(s)  Cases  Label=1, n Label=1, %
  Training 0,1,2,3,4    188          56      29.8%
Validation         5     98          29      29.6%
   Testing         6     65          20      30.8%

Method summary:
          Model                                                          CV selected setting(s) CV probability Validation selected setting(s) Validation probability Testing selected setting(s) Testing probability
       Clinical                                                        LR (random30_rfecv_none)      prob_mean       LR (random30_rfecv_none)              prob_mean    LR (random30_rfecv_none)           prob_mean
    C-radiomics SVM (random0_rfe_top25); SVM (random10_rfe_top25); XGBoost (random10_rfe_top25)      prob_mean   XGBoost (random30_rfe_top17)              prob_mean    SVM (random10_rfe_top20